<sub>**The Annotated ML Handbook** · Part I: Python · by Mahir Absar Khan</sub>

# Chapter 6: Errors, files and modules

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Mahir-606/annotated-ml-handbook/blob/main/notebooks/part-1-python/06-python-errors-files.ipynb) &nbsp; [📚 Contents](../../book/README.md) &nbsp; [🌐 Web edition](https://mahir-606.github.io/annotated-ml-handbook/)

*Run the setup cell first, then read and run the cells from top to bottom. Every code cell is followed by a table explaining what each line does.*

In [1]:
# Repository setup (not part of the book's examples): run this cell first.
import os, sys
from pathlib import Path

# Files the examples create (CSV, models, figures) go into ./outputs next to the notebook.
OUT = Path.cwd() if Path.cwd().name == "outputs" else Path.cwd() / "outputs"
OUT.mkdir(exist_ok=True)
os.chdir(OUT)

# insurance.csv is the practice dataset created in Chapter 11 and reused by later chapters.
# If it is not here yet, copy it from the repository's data/ folder (or download it in Colab).
if not Path("insurance.csv").exists():
    import shutil, urllib.request
    for base in OUT.parents:
        if (base / "data" / "insurance.csv").exists():
            shutil.copy(base / "data" / "insurance.csv", "insurance.csv")
            break
    else:
        try:
            urllib.request.urlretrieve("https://raw.githubusercontent.com/Mahir-606/"
                                       "annotated-ml-handbook/main/data/insurance.csv", "insurance.csv")
        except Exception as e:
            print("Could not fetch insurance.csv; run Chapter 11 first.", e)

# The figure style used in the book.
import matplotlib.pyplot as plt
from cycler import cycler
plt.rcParams.update({
    "figure.figsize": (7.2, 4.0), "figure.dpi": 100,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.alpha": 0.28, "grid.linewidth": 0.6,
    "axes.prop_cycle": cycler(color=["#1f4fff", "#e0245e", "#0f9b62", "#ff6a00",
                                     "#7b3ff2", "#0091b3", "#d99a00", "#d42a2a"]),
    "axes.titleweight": "bold", "axes.titlesize": 12.5, "axes.titlelocation": "left",
    "font.size": 10.5, "legend.frameon": False, "image.cmap": "viridis",
})

# Plotly: interactive in Jupyter/Colab; tools/run_all.py stores static PNGs so GitHub can show them.
try:
    import plotly.io as pio
    pio.renderers.default = os.environ.get("PLOTLY_RENDERER") or (
        "colab" if "google.colab" in sys.modules else "notebook_connected")
except ImportError:
    pass

Real programs read files, call other people's code and run into bad input. This chapter covers handling errors gracefully, reading and writing the file formats you will meet in research, and organising code into modules.

## 6.1 Exceptions and try/except · *Beginner*

> **📖 Definitions**
>
> **Exception**: an object representing an error, which Python *raises* when something goes wrong, such as `ValueError` or `KeyError`. Unhandled, it stops the program.
>
> **Exception handling**: catching an expected error with `try` / `except` so the program can respond instead of crashing.


When something goes wrong Python *raises an exception*. If nothing catches it, the program stops and prints a **traceback**. You can catch exceptions you expect and handle them.

**Catching specific exceptions**

In [2]:
def safe_divide(a, b):
    try:
        result = a / b
    except ZeroDivisionError:
        print("Cannot divide by zero; returning None")
        return None
    except TypeError as err:
        print("Bad types:", err)
        return None
    else:
        return result
    finally:
        print("done:", a, b)

print(safe_divide(10, 4))
print(safe_divide(1, 0))
print(safe_divide("1", 2))

done: 10 4
2.5
Cannot divide by zero; returning None
done: 1 0
None
Bad types: unsupported operand type(s) for /: 'str' and 'int'
done: 1 2
None


<details open><summary><b>What each line does</b></summary>

<table>
<tr><td><code>def safe_divide(a, b):</code></td><td>Define a function <code>safe_divide</code> that takes <code>a</code> and <code>b</code>.</td></tr>
<tr><td><code>&nbsp;&nbsp;&nbsp;&nbsp;try:</code></td><td>Run the risky code inside <code>try</code>.</td></tr>
<tr><td><code>&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;result = a / b</code></td><td>The line that may fail.</td></tr>
<tr><td><code>&nbsp;&nbsp;&nbsp;&nbsp;except ZeroDivisionError:</code></td><td>Runs only if THIS kind of error happened.</td></tr>
<tr><td><code>&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;print(&quot;Cannot divide by zero; returning None&quot;)</code></td><td>Handle the problem gracefully.</td></tr>
<tr><td><code>&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;return None</code></td><td>Send <code>None</code> back to the caller.</td></tr>
<tr><td><code>&nbsp;&nbsp;&nbsp;&nbsp;except TypeError as err:</code></td><td><code>as err</code> binds the exception object so you can inspect it.</td></tr>
<tr><td><code>&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;print(&quot;Bad types:&quot;, err)</code></td><td>The exception object prints as its message.</td></tr>
<tr><td><code>&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;return None</code></td><td>Send <code>None</code> back to the caller.</td></tr>
<tr><td><code>&nbsp;&nbsp;&nbsp;&nbsp;else:</code></td><td>Runs only if no exception occurred.</td></tr>
<tr><td><code>&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;return result</code></td><td>Send <code>result</code> back to the caller.</td></tr>
<tr><td><code>&nbsp;&nbsp;&nbsp;&nbsp;finally:</code></td><td>ALWAYS runs, error or not: use it for clean-up.</td></tr>
<tr><td><code>&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;print(&quot;done:&quot;, a, b)</code></td><td>Printed on every call, even after an error.</td></tr>
<tr><td><code>print(safe_divide(10, 4))</code></td><td>Works: 2.5 (after the finally message).</td></tr>
<tr><td><code>print(safe_divide(1, 0))</code></td><td>Triggers the ZeroDivisionError branch.</td></tr>
<tr><td><code>print(safe_divide(&quot;1&quot;, 2))</code></td><td>Triggers the TypeError branch.</td></tr>
</table>
</details>

> **⚠️ Pitfall** · **Never write a bare `except:`**
>
> `except:` or `except Exception: pass` swallows *every* error, including typos and bugs you need to see. Catch the specific exception you expect, and let everything else crash loudly.

### 6.1.1 Raising your own exceptions · *Intermediate*

Validate inputs early and fail with a clear message. A helpful error message saves hours of debugging later.

**Validating inputs**

In [3]:
class DataValidationError(ValueError):
    pass

def check_probabilities(ps):
    if not ps:
        raise DataValidationError("empty list of probabilities")
    bad = [p for p in ps if not 0 <= p <= 1]
    if bad:
        raise DataValidationError(f"{len(bad)} value(s) outside [0, 1]: {bad[:3]}")
    return True

try:
    check_probabilities([0.2, 1.4, -0.1])
except DataValidationError as e:
    print(type(e).__name__, "->", e)

DataValidationError -> 2 value(s) outside [0, 1]: [1.4, -0.1]


<details open><summary><b>What each line does</b></summary>

<table>
<tr><td><code>class DataValidationError(ValueError):</code></td><td>A custom exception: subclass a built-in one.</td></tr>
<tr><td><code>&nbsp;&nbsp;&nbsp;&nbsp;pass</code></td><td>Do nothing: the body is intentionally empty.</td></tr>
<tr><td><code>def check_probabilities(ps):</code></td><td>Define a function <code>check_probabilities</code> that takes <code>ps</code>.</td></tr>
<tr><td><code>&nbsp;&nbsp;&nbsp;&nbsp;if not ps:</code></td><td>Run the indented block only if <code>not ps</code> is true.</td></tr>
<tr><td><code>&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;raise DataValidationError(&quot;empty list of probabilities&quot;)</code></td><td>Stop with a specific, clear error.</td></tr>
<tr><td><code>&nbsp;&nbsp;&nbsp;&nbsp;bad = [p for p in ps if not 0 &lt;= p &lt;= 1]</code></td><td>Collect every value outside [0, 1].</td></tr>
<tr><td><code>&nbsp;&nbsp;&nbsp;&nbsp;if bad:</code></td><td>Run the indented block only if <code>bad</code> is true.</td></tr>
<tr><td><code>&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;raise DataValidationError(f&quot;{len(bad)} value(s) outside [0, 1]: {bad[:3]}&quot;)</code></td><td>The message says how many are bad and shows a few.</td></tr>
<tr><td><code>&nbsp;&nbsp;&nbsp;&nbsp;return True</code></td><td>Send <code>True</code> back to the caller.</td></tr>
<tr><td><code>try:</code></td><td>Start a block whose errors we want to catch.</td></tr>
<tr><td><code>&nbsp;&nbsp;&nbsp;&nbsp;check_probabilities([0.2, 1.4, -0.1])</code></td><td>Two invalid values.</td></tr>
<tr><td><code>except DataValidationError as e:</code></td><td>Catch only our custom error.</td></tr>
<tr><td><code>&nbsp;&nbsp;&nbsp;&nbsp;print(type(e).__name__, &quot;-&gt;&quot;, e)</code></td><td>The exception&#x27;s class name and its message.</td></tr>
</table>
</details>

`assert` is a quick sanity check for *your own* assumptions while developing:

**assert stops the program when an assumption fails**

⚠️ *This cell raises an error **on purpose**, so you can see the real message.*

In [4]:
n_train, n_labels = 120, 118
assert n_train == n_labels, f"features ({n_train}) and labels ({n_labels}) differ"

AssertionError: features (120) and labels (118) differ

<details open><summary><b>What each line does</b></summary>

<table>
<tr><td><code>n_train, n_labels = 120, 118</code></td><td>Two counts that should match but don&#x27;t.</td></tr>
<tr><td><code>assert n_train == n_labels, f&quot;features ({n_train}) and labels ({n_labels}) differ&quot;</code></td><td>If the condition is False, raises AssertionError with your message.</td></tr>
</table>
</details>

## 6.2 Reading a traceback · *Beginner*

> **📖 Definitions**
>
> **Traceback**: the report Python prints when an exception is not handled: the chain of function calls that led to the error, ending with the error type and message.


Read a traceback **from the bottom up**. The last line names the exception type and message. The lines above show the chain of calls that led there, ending with the line that failed. Part VIII expands on this, with dozens of real examples.

**A traceback through two functions**

⚠️ *This cell raises an error **on purpose**, so you can see the real message.*

In [5]:
def load_scores(record):
    return record["scores"]

def average(record):
    s = load_scores(record)
    return sum(s) / len(s)

average({"score": [1, 2, 3]})

KeyError: 'scores'

<details open><summary><b>What each line does</b></summary>

<table>
<tr><td><code>def load_scores(record):</code></td><td>Define a function <code>load_scores</code> that takes <code>record</code>.</td></tr>
<tr><td><code>&nbsp;&nbsp;&nbsp;&nbsp;return record[&quot;scores&quot;]</code></td><td>Fails here: the dict has no &quot;scores&quot; key.</td></tr>
<tr><td><code>def average(record):</code></td><td>Define a function <code>average</code> that takes <code>record</code>.</td></tr>
<tr><td><code>&nbsp;&nbsp;&nbsp;&nbsp;s = load_scores(record)</code></td><td>Called from here...</td></tr>
<tr><td><code>&nbsp;&nbsp;&nbsp;&nbsp;return sum(s) / len(s)</code></td><td>Send <code>sum(s) / len(s)</code> back to the caller.</td></tr>
<tr><td><code>average({&quot;score&quot;: [1, 2, 3]})</code></td><td>...which was called from here. Note the typo: &quot;score&quot; vs &quot;scores&quot;.</td></tr>
</table>
</details>

## 6.3 Working with files · *Beginner*

> **📖 Definitions**
>
> **File path**: the location of a file, absolute (`/home/ada/data.csv`) or relative to the current working directory (`data/data.csv`).
>
> **Working directory**: the folder a program treats as its starting point for relative paths. Check it with `Path.cwd()`.
>
> **Encoding**: the rule that turns text into bytes on disk. UTF-8 is the modern standard.


Always open files with a `with` block. It guarantees the file is closed even if an error occurs.

**Writing and reading text files**

In [6]:
with open("notes.txt", "w", encoding="utf-8") as f:
    f.write("epoch,loss\n")
    for epoch, loss in enumerate([0.9, 0.6, 0.45], 1):
        f.write(f"{epoch},{loss}\n")

with open("notes.txt", encoding="utf-8") as f:
    header = f.readline().strip()
    rows = [line.strip().split(",") for line in f]
print(header, rows)
with open("notes.txt", "a", encoding="utf-8") as f:
    f.write("4,0.40\n")

epoch,loss [['1', '0.9'], ['2', '0.6'], ['3', '0.45']]


<details open><summary><b>What each line does</b></summary>

<table>
<tr><td><code>with open(&quot;notes.txt&quot;, &quot;w&quot;, encoding=&quot;utf-8&quot;) as f:</code></td><td>Mode &quot;w&quot; = write (creates or overwrites). Always state the encoding.</td></tr>
<tr><td><code>&nbsp;&nbsp;&nbsp;&nbsp;f.write(&quot;epoch,loss\n&quot;)</code></td><td><code>write</code> does not add newlines; include <code>\n</code> yourself.</td></tr>
<tr><td><code>&nbsp;&nbsp;&nbsp;&nbsp;for epoch, loss in enumerate([0.9, 0.6, 0.45], 1):</code></td><td>Loop over <code>enumerate([0.9, 0.6, 0.45], 1)</code>, naming each item <code>epoch, loss</code>.</td></tr>
<tr><td><code>&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;f.write(f&quot;{epoch},{loss}\n&quot;)</code></td><td>One CSV line per epoch.</td></tr>
<tr><td><code>with open(&quot;notes.txt&quot;, encoding=&quot;utf-8&quot;) as f:</code></td><td>Default mode &quot;r&quot; = read.</td></tr>
<tr><td><code>&nbsp;&nbsp;&nbsp;&nbsp;header = f.readline().strip()</code></td><td>Read one line; strip the trailing newline.</td></tr>
<tr><td><code>&nbsp;&nbsp;&nbsp;&nbsp;rows = [line.strip().split(&quot;,&quot;) for line in f]</code></td><td>Iterating a file yields one line at a time (memory-friendly).</td></tr>
<tr><td><code>print(header, rows)</code></td><td>The header string and a list of [epoch, loss] string pairs.</td></tr>
<tr><td><code>with open(&quot;notes.txt&quot;, &quot;a&quot;, encoding=&quot;utf-8&quot;) as f:</code></td><td>Mode &quot;a&quot; = append to the end.</td></tr>
<tr><td><code>&nbsp;&nbsp;&nbsp;&nbsp;f.write(&quot;4,0.40\n&quot;)</code></td><td>Add one more line at the end.</td></tr>
</table>
</details>

### 6.3.1 Paths with pathlib · *Beginner*

`pathlib` handles file paths portably across Windows, macOS and Linux.

**Building, checking and listing paths**

In [7]:
from pathlib import Path

root = Path("project")
data_dir = root / "data" / "raw"
data_dir.mkdir(parents=True, exist_ok=True)
(data_dir / "a.csv").write_text("x\n1\n")
(data_dir / "b.csv").write_text("x\n2\n")
print(sorted(p.name for p in data_dir.glob("*.csv")))
p = data_dir / "a.csv"
print(p.exists(), p.suffix, p.stem, p.parent)

['a.csv', 'b.csv']
True .csv a project/data/raw


<details open><summary><b>What each line does</b></summary>

<table>
<tr><td><code>from pathlib import Path</code></td><td>Import <code>Path</code> from <code>pathlib</code>.</td></tr>
<tr><td><code>root = Path(&quot;project&quot;)</code></td><td>A Path object (no file is created yet).</td></tr>
<tr><td><code>data_dir = root / &quot;data&quot; / &quot;raw&quot;</code></td><td><code>/</code> joins path parts correctly on every operating system.</td></tr>
<tr><td><code>data_dir.mkdir(parents=True, exist_ok=True)</code></td><td>Create the folders, including parents; no error if they exist.</td></tr>
<tr><td><code>(data_dir / &quot;a.csv&quot;).write_text(&quot;x\n1\n&quot;)</code></td><td>Quick write without open().</td></tr>
<tr><td><code>(data_dir / &quot;b.csv&quot;).write_text(&quot;x\n2\n&quot;)</code></td><td>A second small file.</td></tr>
<tr><td><code>print(sorted(p.name for p in data_dir.glob(&quot;*.csv&quot;)))</code></td><td>Find files matching a pattern.</td></tr>
<tr><td><code>p = data_dir / &quot;a.csv&quot;</code></td><td>A path to one file.</td></tr>
<tr><td><code>print(p.exists(), p.suffix, p.stem, p.parent)</code></td><td>Inspect parts of a path.</td></tr>
</table>
</details>

### 6.3.2 JSON and CSV · *Beginner*

> **📖 Definitions**
>
> **CSV (comma-separated values)**: a plain-text table format: one row per line, values separated by commas (or semicolons or tabs).
>
> **JSON**: a plain-text format for nested data made of objects (like dicts), arrays (like lists), strings, numbers, true/false and null.

**JSON for configs and results, CSV for tables**

In [8]:
import json, csv

results = {"model": "rf", "acc": 0.91, "params": {"n_estimators": 300}}
with open("results.json", "w") as f:
    json.dump(results, f, indent=2)
with open("results.json") as f:
    loaded = json.load(f)
print(loaded["params"]["n_estimators"])

rows = [{"id": 1, "group": "A"}, {"id": 2, "group": "B"}]
with open("groups.csv", "w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=["id", "group"])
    writer.writeheader()
    writer.writerows(rows)
with open("groups.csv", newline="") as f:
    print(list(csv.DictReader(f)))

300
[{'id': '1', 'group': 'A'}, {'id': '2', 'group': 'B'}]


<details open><summary><b>What each line does</b></summary>

<table>
<tr><td><code>import json, csv</code></td><td>Import the standard-library <code>json</code> module and the <code>csv</code> module.</td></tr>
<tr><td><code>results = {&quot;model&quot;: &quot;rf&quot;, &quot;acc&quot;: 0.91, &quot;params&quot;: {&quot;n_estimators&quot;: 300}}</code></td><td>A nested dict to save.</td></tr>
<tr><td><code>with open(&quot;results.json&quot;, &quot;w&quot;) as f:</code></td><td>Open a file for writing; it closes automatically.</td></tr>
<tr><td><code>&nbsp;&nbsp;&nbsp;&nbsp;json.dump(results, f, indent=2)</code></td><td>Serialise a dict to a JSON file; indent makes it human-readable.</td></tr>
<tr><td><code>with open(&quot;results.json&quot;) as f:</code></td><td>Open it again for reading.</td></tr>
<tr><td><code>&nbsp;&nbsp;&nbsp;&nbsp;loaded = json.load(f)</code></td><td>Parse it back into Python dicts and lists.</td></tr>
<tr><td><code>print(loaded[&quot;params&quot;][&quot;n_estimators&quot;])</code></td><td>Display the value of <code>loaded[&quot;params&quot;][&quot;n_estimators&quot;]</code>.</td></tr>
<tr><td><code>rows = [{&quot;id&quot;: 1, &quot;group&quot;: &quot;A&quot;}, {&quot;id&quot;: 2, &quot;group&quot;: &quot;B&quot;}]</code></td><td>Rows as a list of dicts.</td></tr>
<tr><td><code>with open(&quot;groups.csv&quot;, &quot;w&quot;, newline=&quot;&quot;) as f:</code></td><td>newline=&quot;&quot; avoids blank lines on Windows.</td></tr>
<tr><td><code>&nbsp;&nbsp;&nbsp;&nbsp;writer = csv.DictWriter(f, fieldnames=[&quot;id&quot;, &quot;group&quot;])</code></td><td>A writer that maps dict keys to columns.</td></tr>
<tr><td><code>&nbsp;&nbsp;&nbsp;&nbsp;writer.writeheader()</code></td><td>Write the header line: id,group.</td></tr>
<tr><td><code>&nbsp;&nbsp;&nbsp;&nbsp;writer.writerows(rows)</code></td><td>Write one line per dict.</td></tr>
<tr><td><code>with open(&quot;groups.csv&quot;, newline=&quot;&quot;) as f:</code></td><td>Re-open the file for reading.</td></tr>
<tr><td><code>&nbsp;&nbsp;&nbsp;&nbsp;print(list(csv.DictReader(f)))</code></td><td>Each row comes back as a dict of STRINGS.</td></tr>
</table>
</details>

> **💡 Tip** · **Use pandas for real tables**
>
> The `csv` module is fine for small jobs, but for analysis you will use `pd.read_csv()` (Part III), which infers types, handles missing values and is much faster.

## 6.4 Modules, packages and imports · *Beginner*

> **📖 Definitions**
>
> **Module**: a single `.py` file containing Python code that can be imported.
>
> **Package / library**: a collection of modules distributed together, such as NumPy or scikit-learn. Installed with `pip` or `conda`.
>
> **Import**: making a module's contents available in your code: `import numpy as np`.


A **module** is any `.py` file; a **package** is a folder of modules. `import` makes their contents available.

**Import styles**

In [9]:
import math
import numpy as np
from statistics import mean, median
from collections import Counter as C
print(math.floor(2.7), np.pi > 3, mean([1, 2, 3]), median([3, 1, 2]), C("aab"))

2 True 2 2 Counter({'a': 2, 'b': 1})


<details open><summary><b>What each line does</b></summary>

<table>
<tr><td><code>import math</code></td><td>Import the module; use names as math.sqrt.</td></tr>
<tr><td><code>import numpy as np</code></td><td>Import with an alias (the community-standard aliases: np, pd, plt, sns, tf).</td></tr>
<tr><td><code>from statistics import mean, median</code></td><td>Import specific names directly.</td></tr>
<tr><td><code>from collections import Counter as C</code></td><td>Import and rename.</td></tr>
<tr><td><code>print(math.floor(2.7), np.pi &gt; 3, mean([1, 2, 3]), median([3, 1, 2]), C(&quot;aab&quot;))</code></td><td>Use a name from each import style.</td></tr>
</table>
</details>

> **⚠️ Pitfall** · **Avoid `from module import *`**
>
> It dumps every name into your namespace, silently overwriting your own variables and making it impossible to tell where a name came from.

Your own module is just a file. Suppose `utils.py` sits next to your script:

**utils.py: a reusable module**

*Shown for reference, not run automatically (it waits for input, needs your own files, or would change your system). Copy it into a cell to try it.*

```python
"""Helper functions for the project."""

def zscore(xs):
    m = sum(xs) / len(xs)
    sd = (sum((x - m) ** 2 for x in xs) / (len(xs) - 1)) ** 0.5
    return [(x - m) / sd for x in xs]

if __name__ == "__main__":
    print(zscore([1, 2, 3]))
```

<details open><summary><b>What each line does</b></summary>

<table>
<tr><td><code>&quot;&quot;&quot;Helper functions for the project.&quot;&quot;&quot;</code></td><td>A module docstring describes the whole file.</td></tr>
<tr><td><code>def zscore(xs):</code></td><td>Any function defined here can be imported elsewhere.</td></tr>
<tr><td><code>&nbsp;&nbsp;&nbsp;&nbsp;m = sum(xs) / len(xs)</code></td><td>Mean.</td></tr>
<tr><td><code>&nbsp;&nbsp;&nbsp;&nbsp;sd = (sum((x - m) ** 2 for x in xs) / (len(xs) - 1)) ** 0.5</code></td><td>Sample standard deviation.</td></tr>
<tr><td><code>&nbsp;&nbsp;&nbsp;&nbsp;return [(x - m) / sd for x in xs]</code></td><td>Send <code>[(x - m) / sd for x in xs]</code> back to the caller.</td></tr>
<tr><td><code>if __name__ == &quot;__main__&quot;:</code></td><td>True only when this file is run directly (python utils.py), not when imported.</td></tr>
<tr><td><code>&nbsp;&nbsp;&nbsp;&nbsp;print(zscore([1, 2, 3]))</code></td><td>A good place for a quick self-test.</td></tr>
</table>
</details>

**main.py: using it**

*Shown for reference, not run automatically (it waits for input, needs your own files, or would change your system). Copy it into a cell to try it.*

```python
from utils import zscore
print(zscore([10, 20, 30]))
```

<details open><summary><b>What each line does</b></summary>

<table>
<tr><td><code>from utils import zscore</code></td><td>Works when utils.py is in the same folder (or on the Python path).</td></tr>
<tr><td><code>print(zscore([10, 20, 30]))</code></td><td>[-1.0, 0.0, 1.0]</td></tr>
</table>
</details>

> **🐞 Bug hunt** · **`ModuleNotFoundError` for your own file**
>
> Python searches the folder of the script being run, then installed packages. Run scripts from the project root, keep imported files beside them, or turn your project into a package with `pip install -e .`. Also check you didn't name your file `numpy.py`, `pandas.py` or `random.py`: it will shadow the real library.

## 6.5 Logging instead of print · *Intermediate*

For long training runs, `logging` gives timestamps and severity levels, and can write to a file.

**Basic logging setup**

In [10]:
import logging, sys
logging.basicConfig(level=logging.INFO, stream=sys.stdout,
                    format="%(asctime)s | %(levelname)s | %(message)s",
                    datefmt="%H:%M:%S", force=True)
log = logging.getLogger("train")
log.info("starting run with lr=%s", 0.001)
log.warning("validation loss increased")
log.debug("not shown: DEBUG is below the INFO level")

19:14:46 | INFO | starting run with lr=0.001


19:14:46 | WARNING | validation loss increased


<details open><summary><b>What each line does</b></summary>

<table>
<tr><td><code>import logging, sys</code></td><td>Import the <code>logging</code> module and the <code>sys</code> module (interpreter information).</td></tr>
<tr><td><code>logging.basicConfig(level=logging.INFO, stream=sys.stdout,</code></td><td>Show INFO and above, printed to the console.</td></tr>
<tr><td><code>&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;format=&quot;%(asctime)s | %(levelname)s | %(message)s&quot;,</code></td><td>Each line shows the time, the level and the message.</td></tr>
<tr><td><code>&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;datefmt=&quot;%H:%M:%S&quot;, force=True)</code></td><td><code>force=True</code> replaces any earlier configuration.</td></tr>
<tr><td><code>log = logging.getLogger(&quot;train&quot;)</code></td><td>A named logger for this part of the program.</td></tr>
<tr><td><code>log.info(&quot;starting run with lr=%s&quot;, 0.001)</code></td><td>Lazy %-style formatting is the logging convention.</td></tr>
<tr><td><code>log.warning(&quot;validation loss increased&quot;)</code></td><td>A WARNING-level message.</td></tr>
<tr><td><code>log.debug(&quot;not shown: DEBUG is below the INFO level&quot;)</code></td><td>Filtered out by level=INFO.</td></tr>
</table>
</details>

> **✍️ Try it yourself**
>
> 1. Write `read_numbers(path)` that returns a list of floats from a text file, skipping lines that can't be converted (catch `ValueError`).
> 2. Save a dict of hyperparameters to `config.json` and load it back.
> 3. Use `Path.glob("**/*.csv")` to count CSV files in a folder tree.
>
> *Worked solutions:* [solutions-parts-1-4.ipynb](../../solutions/solutions-parts-1-4.ipynb), section *Chapter 6*.

---

⬅️ [Chapter 5: Classes and objects](05-python-oop.ipynb) · [📚 Contents](../../book/README.md) · [Chapter 7: The Pythonic toolkit](07-python-toolkit.ipynb) ➡️